# Object-Oriented Programming in Python
## A Hands-On Tutorial

**Duration:** ~2 hours  
**Prerequisites:** Working knowledge of Python (functions, lists, dictionaries, loops, conditionals)  
**Goal:** Learn Object-Oriented Programming (OOP): How to bundle data and behavior into reusable, maintainable classes.

---

### Why OOP?

We constantly work with **entities** that have both **data** and **behavior**:

- A **book** has a title, author, and page count, and can be borrowed or returned.
- A **rectangle** has a width and a height, and we can compute its area.
- A **library** has a collection of books and can be searched.

Trying to model these with only functions and dictionaries quickly becomes messy. OOP lets us bundle the data *and* the operations on that data into a single coherent unit called a **class**.

Throughout this notebook we will mainly use a **`Book`** (and later a small **`Library`**) running example. We will occasionally introduce a second small example (a `Rectangle` or a `Temperature`) when one is more natural for a given topic.

### How to Use This Notebook

Read each section, then **run every code cell yourself**. Try changing parameters, breaking things on purpose, and fixing them. That is how OOP intuition is built. Each section ends with a short exercise.


---
## Table of Contents

0. [**Motivation: why do we need classes?**](#sec-0)
1. [**Classes and instances**](#sec-1)
    - 1.1 Your first class
    - 1.2 Anatomy of the class: what just happened?
    - 1.3 Instances are independent
    - 1.4 A note on attribute creation
    - Exercise 1
2. [**Instance variables vs. class variables**](#sec-2)
    - 2.1 A first example
    - 2.2 The attribute-lookup rule
    - 2.3 The "assignment creates an instance variable" pitfall
    - 2.4 When to use class variables
    - 2.5 The mutable-default trap
    - Exercise 2
3. [**Instance methods, class methods, and static methods**](#sec-3)
    - 3.1 Instance methods (the default)
    - 3.2 Class methods — `@classmethod`
    - 3.3 Static methods — `@staticmethod`
    - 3.4 How to choose?
    - Exercise 3
4. [**Inheritance**](#sec-4)
    - 4.1 Base (parent) class
    - 4.2 Subclass — the simplest form
    - 4.3 Extending the constructor with `super()`
    - 4.4 Overriding methods
    - 4.5 `isinstance` and `issubclass` — polymorphism
    - 4.6 When NOT to use inheritance
    - Exercise 4
5. [**Special methods (dunder methods)**](#sec-5)
    - 5.1 `__repr__` and `__str__`
    - 5.2 Equality: `__eq__` and `__hash__`
    - 5.3 Ordering: `__lt__` and friends
    - 5.4 Arithmetic: `__add__`, `__sub__`, `__mul__`
    - 5.5 Container-like behavior: `__len__`, `__getitem__`, `__iter__`, `__contains__`
    - 5.6 Cheat-sheet of the most useful dunders
    - Exercise 5
6. [**Property decorators (getter, setter, deleter)**](#sec-6)
    - 6.1 The problem
    - 6.2 Getter
    - 6.3 Setter — with validation
    - 6.4 Deleter
    - 6.5 Read-only property
    - 6.6 The golden rule
    - Exercise 6
7. [**Bonus: encapsulation and abstract base classes**](#sec-7)
    - 7.1 Encapsulation and naming conventions
    - 7.2 Abstract base classes (ABCs)
    - 7.3 Composition over inheritance
8. [**Capstone: a mini Library Management System**](#sec-8)
9. [**Summary and further reading**](#sec-9)


---
<a id="sec-0"></a>
## 0. Motivation: Why Do We Need Classes?

Before we meet classes, let us see what life looks like *without* them. Suppose we want to represent books in a library.

### Attempt 1: Separate variables

```python
title_1 = "1984"
author_1 = "George Orwell"
pages_1 = 328

title_2 = "Brave New World"
author_2 = "Aldous Huxley"
pages_2 = 311
```

This does not scale. With 10,000 books, we would have 30,000 loose variables.

### Attempt 2: Dictionaries

```python
book_1 = {"title": "1984", "author": "George Orwell", "pages": 328}
book_2 = {"title": "Brave New World", "author": "Aldous Huxley", "pages": 311}
```

Better, but the *operations* (checking out, returning, searching) live far from the data, and there is nothing preventing us from writing `book_1["titel"] = "1984"` (typo — a silent bug).

### What we really want

- A single **template** that describes what a "book" looks like.
- Every book created from that template automatically has the right structure.
- The **functions that operate on a book live with it**.

That template is called a **class**. Each individual book built from the template is called an **instance** (or **object**).


---
<a id="sec-1"></a>
## 1. Classes and Instances

A **class** is a blueprint. An **instance** is a concrete object built from that blueprint.

Think of it like this:
- `Book` is the class (the concept of "a book").
- `nineteen_eighty_four = Book("1984", "George Orwell", 328)` is an instance (a specific book).

### 1.1 Your first class


In [3]:
from __future__ import annotations  # allows forward references in type hints


class Book:
    """A simple representation of a book."""

    def __init__(self, title: str, author: str, pages: int) -> None:
        # These are instance variables — they belong to THIS particular book.
        self.title: str = title
        self.author: str = author
        self.pages: int = pages

    def summary(self) -> str:
        """Return a short human-readable description of the book."""
        return f"{self.title!r} by {self.author} ({self.pages} pages)"


# Creating instances (objects)
book1 = Book(title="1984", author="George Orwell", pages=328)
book2 = Book(title="Brave New World", author="Aldous Huxley", pages=311)

print(book1.summary())
print(book2.summary())


'1984' by George Orwell (328 pages)
'Brave New World' by Aldous Huxley (311 pages)


### 1.2 What just happened?

Let us dissect the code carefully — every line matters.

**`class Book:`**  
Declares a new class named `Book`. By convention, class names use `PascalCase`.

**`def __init__(self, ...)`**  
A special method called the **constructor** (technically the *initializer*). Python calls it automatically when you write `Book(...)`. Its job is to set up the new instance.

**`self`**  
The first parameter of every instance method is `self`. It is a reference to *the instance being operated on*. You do not pass it explicitly. Python does it for you.  
When you write `book1.summary()`, Python translates it into `Book.summary(book1)` behind the scenes.

**`self.title = title`**  
Attaches an **instance variable** named `title` to the new object. Each instance has its own independent copy.

**Type hints (`title: str`, `-> str`)**  
These are *optional* but highly recommended. They document intent, help IDEs, and are checked by tools like `mypy`. They do **not** enforce types at runtime.

### 1.3 Instances are independent


In [4]:
book1.pages = 350   # modify book1 only
print("book1 pages:", book1.pages)
print("book2 pages:", book2.pages)  # unchanged
print(book1.summary())


book1 pages: 350
book2 pages: 311
'1984' by George Orwell (350 pages)


### 1.4 The `__init__` is not the only way to add attributes

You *can* attach attributes after construction, but this is usually a bad idea because it breaks the contract declared in `__init__`:

```python
book1.language = "English"   # works, but brittle — better to declare it in __init__
```

### Exercise 1

Write a class `Rectangle` with two instance variables: `width` (float) and `height` (float). Add a method `area()` that returns `width * height`, and a method `perimeter()` that returns `2 * (width + height)`.


In [5]:
# Your turn — try to solve before peeking below
class Rectangle:
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


r = Rectangle(width=4.0, height=3.0)
print(f"area = {r.area()}, perimeter = {r.perimeter()}")


area = 12.0, perimeter = 14.0


---
<a id="sec-2"></a>
## 2. Instance Variables vs. Class Variables

So far, every variable we defined lived on the *instance* (via `self.x = ...`). Python also supports **class variables**: values that are **shared by all instances**.

### 2.1 A first example

Every book in a particular library has the same late-return fee policy. That policy is a property of the *class*, not of any one book.


In [6]:
class Book:
    # Class variables — shared across ALL Book instances.
    library_name: str = "City Library"
    late_fee_per_day: float = 0.25   # in USD, same for every book

    def __init__(self, title: str, author: str, pages: int) -> None:
        # Instance variables — unique to each book
        self.title = title
        self.author = author
        self.pages = pages

    def late_fee(self, days_late: int) -> float:
        """Late fee is days-overdue × daily fee."""
        return days_late * self.late_fee_per_day


book1 = Book("1984", "George Orwell", 328)
book2 = Book("Brave New World", "Aldous Huxley", 311)

print("book1.library_name:", book1.library_name)
print("book2.library_name:", book2.library_name)
print("Book.library_name (via class):", Book.library_name)
print(f"book1 late fee for 5 days: ${book1.late_fee(5):.2f}")


book1.library_name: City Library
book2.library_name: City Library
Book.library_name (via class): City Library
book1 late fee for 5 days: $1.25


### 2.2 The attribute-lookup rule

When you write `book1.library_name`, Python first looks on the *instance*. If it does not find it there, it falls back to the *class*. This is why `book1.library_name` returned `"City Library"` even though we never set `library_name` in `__init__`.

### 2.3 A subtle but important pitfall

What happens if we *assign* to `book1.library_name`? It does **not** modify the class variable. It creates a brand new instance variable that shadows it.


In [7]:
book1.library_name = "Central Branch"   # creates an instance variable on book1
print("book1.library_name:", book1.library_name)      # Central Branch (instance wins)
print("book2.library_name:", book2.library_name)      # still City Library
print("Book.library_name:", Book.library_name)        # still City Library


book1.library_name: Central Branch
book2.library_name: City Library
Book.library_name: City Library


To modify the class variable for *everyone*, you must assign through the class itself:


In [8]:
Book.late_fee_per_day = 0.50   # policy change — applies to every Book
print(f"book1 late fee for 5 days: ${book1.late_fee(5):.2f}")
print(f"book2 late fee for 5 days: ${book2.late_fee(5):.2f}")


book1 late fee for 5 days: $2.50
book2 late fee for 5 days: $2.50


### 2.4 When to use class variables

Good use cases:
- **Constants** shared by all instances (e.g. `MAX_CHECKOUT_DAYS = 21`).
- **Default values** that rarely change per-instance.
- **Registries or counters** (we will see one in the exercise).

Bad use cases:
- Mutable defaults such as lists or dicts — they will be *shared* between instances, which is almost always a bug.

### 2.5 The mutable-default trap


In [9]:
class ShelfBUGGY:
    books: list[Book] = []   # BUG: shared by every instance!

    def add(self, book: Book) -> None:
        self.books.append(book)


shelf_a = ShelfBUGGY()
shelf_b = ShelfBUGGY()
shelf_a.add(book1)
print("shelf_a:", [b.title for b in shelf_a.books])
print("shelf_b:", [b.title for b in shelf_b.books])   # also contains 1984 — surprise!


shelf_a: ['1984']
shelf_b: ['1984']


In [10]:
class Shelf:
    def __init__(self) -> None:
        self.books: list[Book] = []   # fresh list per instance

    def add(self, book: Book) -> None:
        self.books.append(book)


shelf_a = Shelf()
shelf_b = Shelf()
shelf_a.add(book1)
print("shelf_a:", [b.title for b in shelf_a.books])
print("shelf_b:", [b.title for b in shelf_b.books])   # correctly empty


shelf_a: ['1984']
shelf_b: []


**Rule of thumb:** put mutable state (lists, dicts, sets) in `__init__`, never as class variables.

### Exercise 2

Add a class variable `num_books` that automatically counts how many `Book` instances have been created. Increment it inside `__init__`.


In [11]:
class Book:
    num_books: int = 0   # class-level counter

    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages
        Book.num_books += 1   # increment the SHARED counter


_ = Book("1984", "George Orwell", 328)
_ = Book("Brave New World", "Aldous Huxley", 311)
_ = Book("Fahrenheit 451", "Ray Bradbury", 249)
print("Total books created:", Book.num_books)


Total books created: 3


---
<a id="sec-3"></a>
## 3. Instance Methods, Class Methods, and Static Methods

Python distinguishes three types of methods. Understanding when to use each is one of the hallmarks of a fluent OOP programmer.

| Kind | First arg | Can access instance? | Can access class? | Typical use |
|------|-----------|----------------------|-------------------|-------------|
| Instance method | `self` | Yes | Yes | Work with instance data |
| Class method | `cls` | No | Yes | Alternative constructors, class-level ops |
| Static method | *(none)* | No | No | Utility functions logically tied to the class |

### 3.1 Instance methods (the default)

These operate on `self` — the individual object.


In [12]:
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    # --- instance method: needs an instance to be called ---
    def reading_time_hours(self, pages_per_hour: float = 40.0) -> float:
        """Estimate how long the book takes to read, in hours."""
        return self.pages / pages_per_hour


book1 = Book("1984", "George Orwell", 328)
print(f"Estimated reading time: {book1.reading_time_hours():.2f} hours")


Estimated reading time: 8.20 hours


### 3.2 Class methods — `@classmethod`

A class method receives the **class itself** as its first argument (by convention named `cls`). The primary use case is an **alternative constructor**: a factory that builds an instance from a different kind of input.

Example: we receive book data as a CSV-style string and want to build a `Book` from it.


In [14]:
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    def summary(self) -> str:
        return f"{self.title!r} by {self.author} ({self.pages} p.)"

    # --- class method: alternative constructor ---
    @classmethod
    def from_csv_row(cls, row: str) -> "Book":
        """Build a Book from a CSV line like '1984,George Orwell,328'."""
        title, author, pages_str = row.split(",")
        return cls(
            title=title.strip(),
            author=author.strip(),
            pages=int(pages_str),
        )

    # --- another classic use: changing class-level state ---
    @classmethod
    def set_default_pages(cls, default: int) -> None:
        cls.default_pages = default


book1 = Book.from_csv_row("1984, George Orwell, 328")
print(book1.summary())

Book.set_default_pages(250)
print("Book.default_pages:", Book.default_pages)


'1984' by George Orwell (328 p.)
Book.default_pages: 250


**Why `cls` and not `Book` directly?** Because when we later subclass `Book`, using `cls` ensures the alternative constructor builds an instance of the *subclass*, not the parent. We will see this in Section 4.

### 3.3 Static methods — `@staticmethod`

A static method belongs to the class **namespace** but does not receive `self` or `cls`. It is essentially a plain function that you have chosen to group with the class because it is logically related.

A good candidate: a helper that validates input *without* needing any instance.


In [13]:
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    @staticmethod
    def is_valid_isbn13(isbn: str) -> bool:
        """A valid ISBN-13 is exactly 13 digits (ignoring hyphens)."""
        cleaned = isbn.replace("-", "")
        return cleaned.isdigit() and len(cleaned) == 13


print(Book.is_valid_isbn13("978-0-452-28423-4"))   # True
print(Book.is_valid_isbn13("1234"))                 # False
print(Book.is_valid_isbn13("abcd-efgh-ijkl-m"))     # False


True
False
False


### 3.4 How to choose?

Ask yourself:
- Do I need data from a specific instance? → **instance method**.
- Do I need the class (e.g., to build an instance or access class variables)? → **class method**.
- Neither? → **static method** (or consider whether it should even be in the class at all).

### Exercise 3

Add to `Rectangle`:
1. A class method `square(side)` that builds a square (a rectangle with equal sides).
2. A static method `is_valid_dimension(x)` that returns `True` if `x > 0`.


In [15]:
class Rectangle:
    def __init__(self, width: float, height: float) -> None:
        if not (self.is_valid_dimension(width) and self.is_valid_dimension(height)):
            raise ValueError("Width and height must be positive")
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    @classmethod
    def square(cls, side: float) -> "Rectangle":
        return cls(width=side, height=side)

    @staticmethod
    def is_valid_dimension(x: float) -> bool:
        return x > 0


sq = Rectangle.square(5.0)
print(f"Square 5x5 area: {sq.area()}")
print("Is 3.0 valid?", Rectangle.is_valid_dimension(3.0))
print("Is -1.0 valid?", Rectangle.is_valid_dimension(-1.0))


Square 5x5 area: 25.0
Is 3.0 valid? True
Is -1.0 valid? False


---
<a id="sec-4"></a>
## 4. Inheritance

Inheritance lets us build a new class on top of an existing one, **reusing** and optionally **extending or overriding** its behavior. It is a direct match for the way many real-world things naturally form hierarchies:

```
Book
 ├─ FictionBook
 └─ Textbook
     └─ ReferenceBook
```

### 4.1 Base (parent) class


In [16]:
from __future__ import annotations


class Book:
    """A generic book."""

    category: str = "general"

    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    def summary(self) -> str:
        return f"[{self.category}] {self.title!r} by {self.author} ({self.pages} p.)"

    def reading_time_hours(self, pages_per_hour: float = 40.0) -> float:
        return self.pages / pages_per_hour


### 4.2 Subclass — the simplest form

To inherit, put the parent name in parentheses.


In [17]:
class FictionBook(Book):
    category = "fiction"

    # We get __init__, summary, and reading_time_hours for free.


novel = FictionBook("1984", "George Orwell", 328)
print(novel.summary())


[fiction] '1984' by George Orwell (328 p.)


### 4.3 Extending the constructor with `super()`

Often we want to add extra attributes in the subclass. We use `super().__init__(...)` to call the parent's constructor.


In [18]:
class Textbook(Book):
    category = "textbook"

    def __init__(
        self,
        title: str,
        author: str,
        pages: int,
        subject: str,
        edition: int,
    ) -> None:
        super().__init__(title=title, author=author, pages=pages)
        self.subject = subject
        self.edition = edition

    def is_latest_edition(self, current_edition: int) -> bool:
        return self.edition == current_edition


tb = Textbook("Calculus", "James Stewart", 1200, subject="Mathematics", edition=8)
print(tb.summary())
print(f"Subject: {tb.subject}, Edition: {tb.edition}")
print("Is edition 8 the latest?", tb.is_latest_edition(8))


[textbook] 'Calculus' by James Stewart (1200 p.)
Subject: Mathematics, Edition: 8
Is edition 8 the latest? True


### 4.4 Overriding methods

A subclass can redefine a method of the parent. Inside the override you may call `super().method()` to reuse the parent's logic.


In [19]:
class ReferenceBook(Textbook):
    """Reference books are not read cover-to-cover, so we override reading time."""

    category = "reference"

    def reading_time_hours(self, pages_per_hour: float = 40.0) -> float:
        # Reference books are skimmed, not read linearly — only ~10% is read.
        return 0.10 * super().reading_time_hours(pages_per_hour)

    def summary(self) -> str:
        base = super().summary()   # reuse the parent's formatting
        return f"{base} | reference (subject: {self.subject})"


ref = ReferenceBook(
    title="Oxford English Dictionary",
    author="Oxford University Press",
    pages=21728,
    subject="Linguistics",
    edition=2,
)
print(ref.summary())
print(f"Effective reading time: {ref.reading_time_hours():.1f} hours")


[reference] 'Oxford English Dictionary' by Oxford University Press (21728 p.) | reference (subject: Linguistics)
Effective reading time: 54.3 hours


### 4.5 `isinstance` and `issubclass`

Inheritance gives you **polymorphism**: the ability to treat objects of different classes uniformly if they share a common parent.


In [20]:
collection: list[Book] = [novel, tb, ref]

total_pages = 0
for book in collection:
    print(book.summary())
    total_pages += book.pages

print(f"\nTotal pages across collection: {total_pages:,}")

# Type checks
print("\nIs novel a FictionBook?", isinstance(novel, FictionBook))
print("Is novel a Book?", isinstance(novel, Book))
print("Is ReferenceBook a subclass of Book?", issubclass(ReferenceBook, Book))


[fiction] '1984' by George Orwell (328 p.)
[textbook] 'Calculus' by James Stewart (1200 p.)
[reference] 'Oxford English Dictionary' by Oxford University Press (21728 p.) | reference (subject: Linguistics)

Total pages across collection: 23,256

Is novel a FictionBook? True
Is novel a Book? True
Is ReferenceBook a subclass of Book? True


### 4.6 When NOT to use inheritance

Inheritance is powerful but easy to overuse. A good heuristic is the **"is-a" vs "has-a" test**:

- A `Textbook` **is a** `Book` → inheritance is appropriate.
- A `Library` **has a** collection of `Book`s → use **composition** (store them as an attribute), not inheritance.

### Exercise 4

Create a class `AudioBook(Book)` that adds a `duration_minutes` attribute and overrides `reading_time_hours` to return `duration_minutes / 60`. Use `super().__init__` to forward the common fields.


In [21]:
class AudioBook(Book):
    category = "audiobook"

    def __init__(
        self,
        title: str,
        author: str,
        pages: int,
        duration_minutes: int,
    ) -> None:
        super().__init__(title=title, author=author, pages=pages)
        self.duration_minutes = duration_minutes

    def reading_time_hours(self, pages_per_hour: float = 40.0) -> float:
        # For an audiobook, listening time is what matters — ignore pages_per_hour.
        return self.duration_minutes / 60


ab = AudioBook("1984", "George Orwell", 328, duration_minutes=660)
print(ab.summary())
print(f"Listening time: {ab.reading_time_hours():.1f} hours")


[audiobook] '1984' by George Orwell (328 p.)
Listening time: 11.0 hours


---
<a id="sec-5"></a>
## 5. Special Methods (Dunder Methods)

**Dunder** = "double underscore". Methods like `__init__`, `__str__`, `__add__`, `__len__` are how Python lets **your** classes integrate seamlessly with the language. When you write `a + b`, `len(x)`, `str(y)`, or `obj[i]`, Python is calling a dunder method under the hood.

Making your classes implement the right dunders is what separates a *merely functional* class from a *pleasant to use* class.

### 5.1 `__repr__` and `__str__`

- `__repr__` → unambiguous, developer-oriented representation (used in the REPL, debugger).
- `__str__` → user-friendly representation (used by `print()` and `str()`).

**Rule:** always implement `__repr__`. Implement `__str__` only if you want a different user-facing format.


In [23]:
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self) -> str:
        return f"Book(title={self.title!r}, author={self.author!r}, pages={self.pages!r})"

    def __str__(self) -> str:
        return f"{self.title!r} by {self.author}"


book = Book("1984", "George Orwell", 328)
print(repr(book))   # developer view
print(str(book))    # user view
print(book)         # print() calls __str__

Book(title='1984', author='George Orwell', pages=328)
'1984' by George Orwell
'1984' by George Orwell


### 5.2 Equality: `__eq__` and `__hash__`

By default, `a == b` checks **identity** (are they the same object in memory?). We usually want **value equality** instead.


In [24]:
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    def __repr__(self) -> str:
        return f"Book({self.title!r}, {self.author!r}, {self.pages})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Book):
            return NotImplemented   # tells Python to try other.__eq__(self)
        return (
            self.title == other.title
            and self.author == other.author
            and self.pages == other.pages
        )

    def __hash__(self) -> int:
        # Objects that compare equal must have equal hashes.
        return hash((self.title, self.author, self.pages))


a = Book("1984", "George Orwell", 328)
b = Book("1984", "George Orwell", 328)
c = Book("1984", "George Orwell", 350)   # different page count

print("a == b?", a == b)   # True — value equality
print("a is b?", a is b)   # False — different objects in memory
print("a == c?", a == c)   # False
print("Set has 2 unique books:", len({a, b, c}))   # a and b collapse


a == b? True
a is b? False
a == c? False
Set has 2 unique books: 2


**Important rule:** if you override `__eq__`, also override `__hash__` (or set `__hash__ = None` if your object should be unhashable, e.g. if it is mutable). Breaking this rule leads to subtle bugs in `set`s and `dict`s.

### 5.3 Ordering: `__lt__`, `__le__`, `__gt__`, `__ge__`

Suppose we want to sort books by number of pages. We implement `__lt__` (less than), and `functools.total_ordering` fills in the rest.


In [25]:
from functools import total_ordering


@total_ordering
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self.pages = pages

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Book):
            return NotImplemented
        return self.pages == other.pages

    def __lt__(self, other: "Book") -> bool:
        if not isinstance(other, Book):
            return NotImplemented
        return self.pages < other.pages

    def __repr__(self) -> str:
        return f"Book({self.title!r}, {self.pages} p.)"


books = [
    Book("1984", "George Orwell", 328),
    Book("War and Peace", "Leo Tolstoy", 1225),
    Book("Fahrenheit 451", "Ray Bradbury", 249),
]

for b in sorted(books):   # ascending by pages thanks to __lt__
    print(b)


Book('Fahrenheit 451', 249 p.)
Book('1984', 328 p.)
Book('War and Peace', 1225 p.)


### 5.4 Arithmetic: `__add__`, `__sub__`, `__mul__`

The clearest example of operator overloading is a simple 2D vector: `v1 + v2` should add componentwise, and `v * 3` should scale.


In [26]:
class Vector2D:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def __add__(self, other: "Vector2D") -> "Vector2D":
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector2D") -> "Vector2D":
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar: float) -> "Vector2D":
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector2D(self.x * scalar, self.y * scalar)

    def __repr__(self) -> str:
        return f"Vector2D({self.x}, {self.y})"


v1 = Vector2D(1, 2)
v2 = Vector2D(3, 4)
print("v1 + v2 =", v1 + v2)
print("v2 - v1 =", v2 - v1)
print("v1 * 3  =", v1 * 3)


v1 + v2 = Vector2D(4, 6)
v2 - v1 = Vector2D(2, 2)
v1 * 3  = Vector2D(3, 6)


**Immutability tip:** arithmetic dunders should return a *new* object, not mutate `self`. This matches how `3 + 5` does not modify `3`.

### 5.5 Container-like behavior: `__len__`, `__getitem__`, `__iter__`, `__contains__`

Let's make a `Shelf` of books that behaves like a list — you can call `len()` on it, index into it, iterate over it, and use `in`.


In [27]:
from collections.abc import Iterator


class Shelf:
    def __init__(self, books: list[Book] | None = None) -> None:
        self.books: list[Book] = list(books) if books else []

    def add(self, book: Book) -> None:
        self.books.append(book)

    def __len__(self) -> int:
        return len(self.books)

    def __getitem__(self, index: int) -> Book:
        return self.books[index]

    def __iter__(self) -> Iterator[Book]:
        return iter(self.books)

    def __contains__(self, title: str) -> bool:
        return any(b.title == title for b in self.books)


shelf = Shelf()
shelf.add(Book("1984", "George Orwell", 328))
shelf.add(Book("Brave New World", "Aldous Huxley", 311))
shelf.add(Book("Fahrenheit 451", "Ray Bradbury", 249))

print("len:", len(shelf))
print("first book:", shelf[0])

print("\niterate:")
for b in shelf:
    print("  ", b.title)

print("\n'1984' in shelf?", "1984" in shelf)
print("'Moby Dick' in shelf?", "Moby Dick" in shelf)


len: 3
first book: Book('1984', 328 p.)

iterate:
   1984
   Brave New World
   Fahrenheit 451

'1984' in shelf? True
'Moby Dick' in shelf? False


### 5.7 A cheat-sheet of the most useful dunders

| Dunder | Triggered by | Typical use |
|---|---|---|
| `__init__` | `Cls(...)` | Initialize instance |
| `__repr__` | `repr(x)`, REPL | Debug-friendly string |
| `__str__` | `str(x)`, `print(x)` | User-friendly string |
| `__eq__`, `__hash__` | `==`, `set`, `dict` | Value equality |
| `__lt__`, ... | `<`, `sorted` | Ordering |
| `__add__`, `__sub__`, `__mul__` | `+`, `-`, `*` | Arithmetic |
| `__len__` | `len(x)` | Size |
| `__getitem__`, `__setitem__` | `x[i]`, `x[i] = v` | Indexing |
| `__iter__`, `__next__` | `for ... in x` | Iteration |
| `__contains__` | `v in x` | Membership |
| `__call__` | `x(...)` | Make instance callable |
| `__enter__`, `__exit__` | `with x: ...` | Context manager |

### Exercise 5

Give `Rectangle` a meaningful `__repr__`, an `__eq__` that compares width and height, and an `__lt__` that compares rectangles by area (so `sorted(...)` sorts them smallest-first).


---
<a id="sec-6"></a>
## 6. Property Decorators (Getter, Setter, Deleter)

### 6.1 The problem

Say we have a `Temperature` that stores a value in Celsius. We want:
1. **Read-only** access to a *computed* value (temperature in Fahrenheit).
2. **Validation** when someone assigns a new temperature (reject values below absolute zero).

A naive first attempt would be a method `get_celsius()`. But then users write `t.get_celsius()` instead of `t.celsius` — ugly. And we cannot validate attribute assignment.

**Properties solve this elegantly.** They make a *method* look like an *attribute*.

### 6.2 Getter


In [28]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        self._celsius = celsius   # note the leading underscore

    @property
    def celsius(self) -> float:
        """Temperature in Celsius."""
        return self._celsius

    @property
    def fahrenheit(self) -> float:
        """Computed on the fly — behaves like an attribute."""
        return self._celsius * 9 / 5 + 32


t = Temperature(100.0)
print("celsius:", t.celsius)           # no parentheses — looks like an attribute
print("fahrenheit:", t.fahrenheit)


celsius: 100.0
fahrenheit: 212.0


Notice that `celsius` and `fahrenheit` are **accessed without parentheses**. The `@property` decorator transforms the method into a computed attribute.

The **leading underscore** in `self._celsius` is a Python convention meaning "this is an internal implementation detail; do not touch it from outside."

### 6.3 Setter — with validation

Now let's add a setter that validates: you can't have a temperature below absolute zero (−273.15 °C).


In [29]:
class Temperature:
    ABS_ZERO_C: float = -273.15

    def __init__(self, celsius: float) -> None:
        self.celsius = celsius   # goes through the setter — validation kicks in even here

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < self.ABS_ZERO_C:
            raise ValueError(f"Temperature {value} °C is below absolute zero")
        self._celsius = value


t = Temperature(25.0)
print("celsius:", t.celsius)

t.celsius = 30.0         # triggers the setter — OK
print("new celsius:", t.celsius)

try:
    t.celsius = -500     # triggers the setter — fails
except ValueError as e:
    print("Caught:", e)


celsius: 25.0
new celsius: 30.0
Caught: Temperature -500 °C is below absolute zero


Three things to notice:
1. The setter is defined with `@celsius.setter` — the decorator name matches the property name.
2. Setting `self.celsius = celsius` inside `__init__` goes through the setter too. This is the **idiomatic** way to get validation at construction without duplicating code.
3. Users still write `t.celsius = 30.0` — the syntax is unchanged. Properties are **transparent**.

### 6.4 Deleter

Less common, but occasionally useful (for example, clearing a cache).


In [30]:
class Temperature:
    ABS_ZERO_C: float = -273.15

    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < self.ABS_ZERO_C:
            raise ValueError(f"Temperature {value} °C is below absolute zero")
        self._celsius = value

    @celsius.deleter
    def celsius(self) -> None:
        print("Deleting temperature reading")
        del self._celsius


t = Temperature(25.0)
del t.celsius    # triggers the deleter
try:
    print(t.celsius)
except AttributeError as e:
    print("Caught:", e)


Deleting temperature reading
Caught: 'Temperature' object has no attribute '_celsius'


### 6.5 Read-only property

If you define *only* a getter (no setter), the property becomes **read-only**. This is perfect for *derived* quantities.


In [31]:
class Rectangle:
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height

    @property
    def area(self) -> float:
        """Derived — no setter, so read-only."""
        return self.width * self.height


r = Rectangle(4.0, 3.0)
print("area:", r.area)
try:
    r.area = 999   # no setter defined → AttributeError
except AttributeError as e:
    print("Caught:", e)


area: 12.0
Caught: property 'area' of 'Rectangle' object has no setter


### 6.6 The golden rule

> **Start with plain attributes.** Only reach for `@property` when you need *computed values*, *validation*, or *encapsulation of internal state*. You do not need a getter/setter for every attribute — that is Java thinking, not Python thinking.

### Exercise 6

Extend `Book` so that `pages` is a property that refuses to be set to a non-positive value. Add a read-only property `reading_time_hours` that returns `pages / 40`.


In [34]:
class Book:
    def __init__(self, title: str, author: str, pages: int) -> None:
        self.title = title
        self.author = author
        self._pages = pages   # goes through the setter

    @property
    def pages(self) -> int:
        return self._pages

    @pages.setter
    def pages(self, value: int) -> None:
        if value <= 0:
            raise ValueError(f"Pages must be positive, got {value}")
        self._pages = value

    @property
    def reading_time_hours(self) -> float:
        return self._pages / 40


b = Book("1984", "George Orwell", 328)
print(f"reading_time_hours: {b.reading_time_hours:.2f}")
try:
    b.pages = -5
except ValueError as e:
    print("Caught:", e)


reading_time_hours: 8.20
Caught: Pages must be positive, got -5


---
<a id="sec-7"></a>
## 7. Bonus: Encapsulation and Abstract Base Classes

### 7.1 Encapsulation and naming conventions

Python does not have strict `private` / `protected` keywords like Java or C++. Instead, it relies on **conventions**:

| Prefix | Meaning | Example |
|---|---|---|
| no prefix | public API | `self.title` |
| `_single` | "internal — use at your own risk" | `self._celsius` |
| `__double` | name-mangled, strongly discouraged access | `self.__secret` |

A leading underscore is a *social contract*: the code will still let you read `t._celsius`, but you are signaling to yourself and to reviewers "this is subject to change, do not rely on it".

### 7.2 Abstract base classes (ABCs)

Sometimes you want to say "every subclass **must** implement this method". The `abc` module makes this explicit: you cannot instantiate an abstract class, and a subclass that forgets to override an abstract method also becomes abstract.


In [35]:
from abc import ABC, abstractmethod
import math


class Shape(ABC):
    """Every concrete shape must know its own area and perimeter."""

    @abstractmethod
    def area(self) -> float:
        ...

    @abstractmethod
    def perimeter(self) -> float:
        ...

    def describe(self) -> str:
        return (
            f"{type(self).__name__}: "
            f"area={self.area():.3f}, perimeter={self.perimeter():.3f}"
        )


class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


class Circle(Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius


# Cannot instantiate abstract class:
try:
    Shape()
except TypeError as e:
    print("Caught:", e)

for shape in [Rectangle(4, 3), Circle(5)]:
    print(shape.describe())


Caught: Can't instantiate abstract class Shape with abstract methods area, perimeter
Rectangle: area=12.000, perimeter=14.000
Circle: area=78.540, perimeter=31.416


### 7.3 Composition over inheritance

A `Library` *has* books — it is not *a* specific kind of book. This is **composition**, and it is usually a cleaner design than trying to inherit.


In [36]:
class Library:
    def __init__(self, name: str) -> None:
        self.name = name
        self._books: list[Book] = []

    def add(self, book: Book) -> None:
        self._books.append(book)

    @property
    def total_books(self) -> int:
        return len(self._books)

    def __iter__(self):
        return iter(self._books)

    def __repr__(self) -> str:
        return f"Library({self.name!r}, {self.total_books} books)"


lib = Library("City Library")
lib.add(Book("1984", "George Orwell", 328))
lib.add(Book("Brave New World", "Aldous Huxley", 311))
print(lib)


Library('City Library', 2 books)


---
<a id="sec-8"></a>
## 8. Capstone: A Mini Library Management System

Now we combine **everything**: inheritance, abstract methods, dunders, properties, class methods, and static methods to design a small but realistic library management system.

Design goals:
- A common `LibraryItem` abstract base class with a common interface.
- Concrete subclasses `PrintBook`, `EBook`, and `AudioBook`.
- A `Library` class that *contains* items (composition), supports iteration, membership testing, and length.
- Validation of user input via properties.
- Value equality and ordering via dunders.


In [37]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from functools import total_ordering


@total_ordering
class LibraryItem(ABC):
    """Abstract base for any item a library might lend."""

    _id_counter: int = 0   # class-level counter for auto-assigning IDs

    def __init__(self, title: str, author: str) -> None:
        LibraryItem._id_counter += 1
        self.item_id: int = LibraryItem._id_counter
        self.title = title
        self.author = author
        self._is_checked_out: bool = False

    # --- abstract: every concrete item must say how long it takes to "read" ---
    @abstractmethod
    def estimated_time_hours(self) -> float: ...

    # --- shared lending behavior ---
    def check_out(self) -> None:
        if self._is_checked_out:
            raise RuntimeError(f"{self.title!r} is already checked out")
        self._is_checked_out = True

    def return_item(self) -> None:
        if not self._is_checked_out:
            raise RuntimeError(f"{self.title!r} was not checked out")
        self._is_checked_out = False

    @property
    def is_available(self) -> bool:
        return not self._is_checked_out

    # --- dunders for nice UX ---
    def __repr__(self) -> str:
        status = "available" if self.is_available else "checked out"
        return (
            f"{type(self).__name__}(id={self.item_id}, "
            f"title={self.title!r}, author={self.author!r}, {status})"
        )

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, LibraryItem):
            return NotImplemented
        return self.item_id == other.item_id

    def __hash__(self) -> int:
        return hash(self.item_id)

    def __lt__(self, other: "LibraryItem") -> bool:
        if not isinstance(other, LibraryItem):
            return NotImplemented
        return self.estimated_time_hours() < other.estimated_time_hours()


class PrintBook(LibraryItem):
    PAGES_PER_HOUR: int = 40   # class variable, typical reading speed

    def __init__(self, title: str, author: str, pages: int) -> None:
        super().__init__(title, author)
        self._pages = pages   # validated via setter

    @property
    def pages(self) -> int:
        return self._pages

    @pages.setter
    def pages(self, value: int) -> None:
        if value <= 0:
            raise ValueError(f"pages must be positive, got {value}")
        self._pages = value

    def estimated_time_hours(self) -> float:
        return self._pages / self.PAGES_PER_HOUR


class EBook(PrintBook):
    """An e-book: same reading-time model as a print book, plus a file size."""

    def __init__(self, title: str, author: str, pages: int, file_size_mb: float) -> None:
        super().__init__(title, author, pages)
        self.file_size_mb = file_size_mb


class AudioBook(LibraryItem):
    def __init__(self, title: str, author: str, duration_minutes: int) -> None:
        super().__init__(title, author)
        self._duration_minutes = duration_minutes   # validated via setter

    @property
    def duration_minutes(self) -> int:
        return self._duration_minutes

    @duration_minutes.setter
    def duration_minutes(self, value: int) -> None:
        if value <= 0:
            raise ValueError(f"duration_minutes must be positive, got {value}")
        self._duration_minutes = value

    def estimated_time_hours(self) -> float:
        return self._duration_minutes / 60


@dataclass
class Library:
    """A library that HAS items (composition, not inheritance)."""

    name: str
    _items: list[LibraryItem] = field(default_factory=list)

    # --- alternative constructor ---
    @classmethod
    def from_items(cls, name: str, items: list[LibraryItem]) -> "Library":
        lib = cls(name=name)
        for it in items:
            lib.add(it)
        return lib

    # --- static helper ---
    @staticmethod
    def is_valid_name(name: str) -> bool:
        return isinstance(name, str) and len(name.strip()) > 0

    def add(self, item: LibraryItem) -> None:
        self._items.append(item)

    def find_by_title(self, title: str) -> list[LibraryItem]:
        return [it for it in self._items if it.title.lower() == title.lower()]

    @property
    def available_items(self) -> list[LibraryItem]:
        return [it for it in self._items if it.is_available]

    # --- container dunders ---
    def __len__(self) -> int:
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

    def __contains__(self, title: str) -> bool:
        return bool(self.find_by_title(title))

    def __repr__(self) -> str:
        return f"Library({self.name!r}, {len(self)} items, {len(self.available_items)} available)"


# --- Demo ---
lib = Library.from_items(
    name="Central Library",
    items=[
        PrintBook("1984", "George Orwell", 328),
        EBook("Brave New World", "Aldous Huxley", 311, file_size_mb=1.2),
        AudioBook("Fahrenheit 451", "Ray Bradbury", duration_minutes=300),
    ],
)

print(lib)
print()

# Iterate (__iter__) and format (__repr__)
for item in lib:
    print(f"  {item}  ~{item.estimated_time_hours():.1f} h")

# Membership (__contains__)
print("\n'1984' in library?", "1984" in lib)

# Check out an item, show availability change
item = lib.find_by_title("1984")[0]
item.check_out()
print("\nAfter checkout:")
print(lib)

# Sort by estimated time (__lt__)
print("\nSorted by estimated reading/listening time:")
for it in sorted(lib):
    print(f"  {it.title:<20}  {it.estimated_time_hours():.1f} h")

# Try to instantiate the abstract base — will fail
try:
    LibraryItem("x", "y")
except TypeError as e:
    print("\nCannot instantiate abstract class:", e)


Library('Central Library', 3 items, 3 available)

  PrintBook(id=1, title='1984', author='George Orwell', available)  ~8.2 h
  EBook(id=2, title='Brave New World', author='Aldous Huxley', available)  ~7.8 h
  AudioBook(id=3, title='Fahrenheit 451', author='Ray Bradbury', available)  ~5.0 h

'1984' in library? True

After checkout:
Library('Central Library', 3 items, 2 available)

Sorted by estimated reading/listening time:
  Fahrenheit 451        5.0 h
  Brave New World       7.8 h
  1984                  8.2 h

Cannot instantiate abstract class: Can't instantiate abstract class LibraryItem with abstract method estimated_time_hours


---
<a id="sec-9"></a>
## 9. Summary and Further Reading

### Key ideas

1. **A class is a blueprint; an instance is a concrete object built from it.** `self` refers to the instance.
2. **Instance variables** are per-object (`self.x`). **Class variables** are shared across all instances — great for constants, dangerous for mutable defaults.
3. **Three method types:** instance methods (`self`), class methods (`@classmethod`, `cls`, alternative constructors), and static methods (`@staticmethod`, namespace-only utilities).
4. **Inheritance** expresses "is-a" relationships. Use `super()` to call parent behavior. Prefer **composition** ("has-a") when in doubt.
5. **Dunder methods** make your classes feel native — implement `__repr__`, `__eq__`/`__hash__`, and the operators that matter to your domain.
6. **Properties** give you attribute syntax with method power — validation, computed values, and encapsulation.
7. **Abstract base classes** enforce interfaces for collaborators.
8. **Immutability (e.g. `@dataclass(frozen=True)`) and validation in `__init__` or setters** prevent whole classes of bugs.

### Mental checklist when designing a class

- What is the object *responsible* for? (Single Responsibility Principle)
- What data must it hold? → instance variables.
- What operations make sense on it? → instance methods.
- Are there alternative ways to construct it? → class methods.
- Does it need to interoperate with Python syntax (`+`, `==`, `for`, `len`, ...)? → dunders.
- Are some attributes derived or must they be validated? → properties.
- Is there a hierarchy of related types? → inheritance (and maybe an ABC).

### Recommended next steps

- The [`dataclasses`](https://docs.python.org/3/library/dataclasses.html) module — auto-generates `__init__`, `__repr__`, `__eq__` from type hints. Great for value objects.
- `typing` and `mypy` for static type-checking.
- The `abc` module and `typing.Protocol` for defining interfaces.
- Python Data Model documentation: https://docs.python.org/3/reference/datamodel.html — the definitive reference on dunders.
- *Fluent Python* by Luciano Ramalho — an excellent reference for idiomatic, advanced Python OOP.

Happy coding!
